# PipelineWatch-NG — Module 1: Data Ingestion
## Satellite-based crude oil theft monitoring · Niger Delta, Nigeria

**Study area:** Trans Niger Pipeline (TNP) corridor  
**Bounding box:** 5.0°–5.8°N, 6.5°–7.2°E  

### What this notebook does
1. Authenticates with Google Earth Engine (zero downloads)
2. Ingests Sentinel-1 SAR imagery → detects oil-spill dark spots  
3. Ingests FIRMS/VIIRS thermal data → detects illegal refinery fire hotspots  
4. Ingests TROPOMI SO₂ data → detects chemical plumes from crude burning  
5. Builds an interactive risk map you can share  
6. Exports GeoJSON outputs for the Module 4 dashboard

### All-weather capability
| Sensor | Cloud-penetrating | Night-capable |
|--------|:-----------------:|:-------------:|
| Sentinel-1 SAR | ✅ C-band radar | ✅ Active sensor |
| FIRMS/VIIRS | ✅ Thermal IR | ✅ Thermal IR |
| TROPOMI SO₂ | ✅ UV backscatter | ❌ Daytime only |

---

## Cell 1 — Environment check
Run this first to confirm all packages are installed correctly.

In [ ]:
import importlib

pkg_map = {
    "earthengine-api": "ee",
    "geemap": "geemap",
    "geopandas": "geopandas",
    "folium": "folium",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
}

missing = []
for pkg, imp in pkg_map.items():
    try:
        importlib.import_module(imp)
        print(f"  OK  {pkg}")
    except ImportError:
        missing.append(pkg)
        print(f"  MISSING  {pkg}")

if missing:
    print(f"\nInstall with: pip install {chr(32).join(missing)}")
else:
    print("\nAll packages installed.")

## Cell 2 — GEE Authentication

**First time only:** Opens a browser tab to sign in with Google. After that, credentials are saved to  and this step is automatic.

1. Create a free GEE project at https://code.earthengine.google.com
2. Replace  below with your actual Project ID

In [ ]:
import ee

# ── CONFIGURE THIS ──────────────────────────────────────────────────────────
GEE_PROJECT_ID = "your-project-id"   # e.g. "pipelinewatch-ng-demo"
# ────────────────────────────────────────────────────────────────────────────

try:
    ee.Initialize(project=GEE_PROJECT_ID)
    test = ee.Number(42).getInfo()
    print(f"GEE initialised — project: {GEE_PROJECT_ID}")
    print(f"Connection test: {test}  (should be 42)")
except ee.EEException:
    print("Not authenticated. Running ee.Authenticate()...")
    print("A browser tab will open — sign in and paste the token below.")
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)
    print("Authentication successful.")

## Cell 3 — Study area setup

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from config.roi_gee import (
    ROI_BOUNDS, ROI_CENTER, ROI_ZOOM,
    BASELINE_START, BASELINE_END,
    RECENT_START, RECENT_END,
    SO2_THRESHOLD_DU, SAR_DARK_SPOT_SIGMA, FIRMS_BRIGHTNESS_K
)

ROI = ee.Geometry.Rectangle(ROI_BOUNDS)

print("Study area: Trans Niger Pipeline (TNP) corridor")
print(f"Bounding box: {ROI_BOUNDS}")
print(f"Baseline:     {BASELINE_START} -> {BASELINE_END}")
print(f"Recent:       {RECENT_START} -> {RECENT_END}")
print()
print("Thresholds:")
print(f"  SAR dark spot:   mean - {SAR_DARK_SPOT_SIGMA}sigma")
print(f"  FIRMS T21:       > {FIRMS_BRIGHTNESS_K} K")
print(f"  TROPOMI SO2:     > {SO2_THRESHOLD_DU} Dobson Units")

## Cell 4 — Sentinel-1 SAR ingestion and processing

Runs entirely server-side on Google. No data downloaded to your machine.

**Physics:** Oil on water dampens capillary waves, reducing Bragg scattering of radar pulses. Oil slicks appear as anomalously DARK patches in VV polarisation imagery.

In [ ]:
from modules.m1_ingestion.gee_sentinel1 import (
    build_s1_composite, extract_dark_spot_vectors, get_s1_viz_params
)

print("=== Sentinel-1 SAR Ingestion ===")

print(f"Building BASELINE composite ({BASELINE_START} -> {BASELINE_END})...")
s1_baseline, s1_baseline_col, n_baseline = build_s1_composite(
    ROI, BASELINE_START, BASELINE_END,
    apply_filter=True, sigma_threshold=SAR_DARK_SPOT_SIGMA
)

print(f"\nBuilding RECENT composite ({RECENT_START} -> {RECENT_END})...")
s1_recent, s1_recent_col, n_recent = build_s1_composite(
    ROI, RECENT_START, RECENT_END,
    apply_filter=True, sigma_threshold=SAR_DARK_SPOT_SIGMA
)

print(f"\nScenes used — baseline: {n_baseline}, recent: {n_recent}")

print("\nExtracting dark spot candidate polygons...")
dark_spots = extract_dark_spot_vectors(s1_recent, ROI, min_area_m2=10000)
n_spots = dark_spots.size().getInfo()
print(f"Dark spot candidate zones: {n_spots}")

## Cell 5 — FIRMS/VIIRS fire hotspot ingestion

In [ ]:
from modules.m1_ingestion.gee_fire_gas import (
    get_firms_collection, compute_firms_composite,
    extract_fire_hotspots, get_firms_viz_params
)

print("=== FIRMS / VIIRS Fire Hotspot Ingestion ===")

print(f"Fetching BASELINE fire data ({BASELINE_START} -> {BASELINE_END})...")
firms_baseline_col = get_firms_collection(ROI, BASELINE_START, BASELINE_END)
n_firms_base = firms_baseline_col.size().getInfo()
print(f"  FIRMS images found: {n_firms_base}")
firms_baseline_comp = compute_firms_composite(firms_baseline_col, ROI)

print(f"\nFetching RECENT fire data ({RECENT_START} -> {RECENT_END})...")
firms_recent_col = get_firms_collection(ROI, RECENT_START, RECENT_END)
n_firms_rec = firms_recent_col.size().getInfo()
print(f"  FIRMS images found: {n_firms_rec}")
firms_recent_comp = compute_firms_composite(firms_recent_col, ROI)

print("\nExtracting persistent fire hotspots...")
fire_hotspots = extract_fire_hotspots(
    firms_recent_comp, ROI,
    t21_threshold_k=FIRMS_BRIGHTNESS_K, min_count=3
)
n_hotspots = fire_hotspots.size().getInfo()
print(f"Persistent fire hotspots: {n_hotspots}")

if n_hotspots > 0:
    sample = fire_hotspots.limit(3).getInfo()
    print("\nSample attributes:")
    for feat in sample["features"]:
        p = feat["properties"]
        print(f"  T21={p.get(chr(84)+chr(50)+chr(49)+chr(95)+chr(109)+chr(97)+chr(120)+chr(95)+chr(75), chr(63)):.0f}K  "
              f"count={p.get(chr(102)+chr(105)+chr(114)+chr(101)+chr(95)+chr(99)+chr(111)+chr(117)+chr(110)+chr(116), 0):.0f}  "
              f"source={p.get(chr(108)+chr(105)+chr(107)+chr(101)+chr(108)+chr(121)+chr(95)+chr(115)+chr(111)+chr(117)+chr(114)+chr(99)+chr(101), chr(63))}")

## Cell 6 — TROPOMI SO₂ ingestion

This is the all-weather chemical fingerprint of illegal crude burning. Persistent SO₂ > 3 Dobson Units co-located with fire = high-confidence illegal refinery.

In [ ]:
from modules.m1_ingestion.gee_fire_gas import (
    get_tropomi_so2_collection, compute_so2_composite,
    extract_so2_anomalies, compute_fire_gas_risk_score, get_so2_viz_params
)

print("=== TROPOMI SO2 Ingestion ===")

print(f"Fetching BASELINE SO2 ({BASELINE_START} -> {BASELINE_END})...")
tropomi_base_col = get_tropomi_so2_collection(ROI, BASELINE_START, BASELINE_END)
n_tropomi_base = tropomi_base_col.size().getInfo()
print(f"  Images found: {n_tropomi_base}")
so2_baseline = compute_so2_composite(tropomi_base_col, ROI)

print(f"\nFetching RECENT SO2 ({RECENT_START} -> {RECENT_END})...")
tropomi_rec_col = get_tropomi_so2_collection(ROI, RECENT_START, RECENT_END)
n_tropomi_rec = tropomi_rec_col.size().getInfo()
print(f"  Images found: {n_tropomi_rec}")
so2_recent = compute_so2_composite(tropomi_rec_col, ROI)

print(f"\nExtracting SO2 anomaly zones (threshold: {SO2_THRESHOLD_DU} DU)...")
so2_anomalies = extract_so2_anomalies(
    so2_recent, ROI, threshold_du=SO2_THRESHOLD_DU, min_obs=5
)
n_so2 = so2_anomalies.size().getInfo()
print(f"SO2 anomaly zones: {n_so2}")

print("\nComputing combined fire + SO2 risk score...")
fire_gas_score = compute_fire_gas_risk_score(
    so2_recent, firms_recent_comp, ROI,
    so2_threshold=SO2_THRESHOLD_DU, t21_threshold=FIRMS_BRIGHTNESS_K
)
print("Risk score ready (0=none, 1=fire only, 2=SO2 only, 3=fire+SO2)")

## Cell 7 — Summary table

In [ ]:
import pandas as pd
from datetime import datetime

summary = pd.DataFrame([
    {"Sensor": "Sentinel-1 SAR", "Period": f"{RECENT_START} to {RECENT_END}",
     "Scenes": n_recent, "Detections": n_spots, "Type": "Oil spill dark spots",
     "All-weather": "Yes", "Night": "Yes"},
    {"Sensor": "FIRMS/VIIRS", "Period": f"{RECENT_START} to {RECENT_END}",
     "Scenes": n_firms_rec, "Detections": n_hotspots, "Type": "Fire hotspots",
     "All-weather": "Partial", "Night": "Yes"},
    {"Sensor": "TROPOMI SO2", "Period": f"{RECENT_START} to {RECENT_END}",
     "Scenes": n_tropomi_rec, "Detections": n_so2, "Type": "SO2 anomalies",
     "All-weather": "Partial", "Night": "No"},
])
print("=== Module 1 Ingestion Summary ===")
print(summary.to_string(index=False))
print(f"\nGenerated: {datetime.now().strftime(chr(37)+chr(89)+chr(45)+chr(37)+chr(109)+chr(45)+chr(37)+chr(100)+chr(32)+chr(37)+chr(72)+chr(58)+chr(37)+chr(77))}")

## Cell 8 — Interactive risk map

All three sensor layers on a satellite basemap. Use the layer control (top-right) to toggle.

> **Tip:** Zoom into creek networks — look for SAR dark spots near waterways and co-located SO₂ elevation.

In [ ]:
import geemap.foliumap as geemap

s1_viz   = get_s1_viz_params()
firms_viz = get_firms_viz_params()
so2_viz  = get_so2_viz_params()

m = geemap.Map(center=ROI_CENTER, zoom=ROI_ZOOM)
m.add_basemap("SATELLITE")

m.addLayer(s1_recent.select("VV"), s1_viz["VV"],
           "S1 VV backscatter (dB)")
m.addLayer(s1_recent.select("dark_spot_mask").selfMask(),
           s1_viz["dark_spot_mask"], "SAR dark spots — oil candidates")
m.addLayer(firms_recent_comp.select("T21_max"), firms_viz["T21_max"],
           "VIIRS max brightness temp (K)")
m.addLayer(so2_recent.select("SO2_mean_DU"), so2_viz["SO2_mean_DU"],
           "TROPOMI SO2 mean (Dobson Units)")
m.addLayer(fire_gas_score.selfMask(),
           {"min": 1, "max": 3, "palette": ["FAC775", "EF9F27", "E24B4A"], "opacity": 0.75},
           "Fire + SO2 combined risk score")
m.addLayer(ee.FeatureCollection([ee.Feature(ROI)]),
           {"color": "185FA5", "fillColor": "00000000"},
           "TNP corridor study area")

m.add_layer_control()
m.add_legend(
    title="PipelineWatch-NG Signals",
    legend_dict={
        "SAR dark spot (oil)": "E24B4A",
        "VIIRS fire hotspot":  "EF9F27",
        "TROPOMI SO2 anomaly": "1D9E75",
        "High risk (fire+SO2)": "A32D2D",
    },
    position="bottomright"
)
print("Map rendered below — use layer control to toggle signals.")
m

## Cell 9 — Fire hotspot time series

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np
import os

def get_monthly_fire_counts(collection, roi, threshold_k=330.0):
    def count_pixels(image):
        date  = image.date().format("YYYY-MM-dd")
        hot   = image.select("T21").gt(threshold_k)
        count = hot.reduceRegion(
            reducer=ee.Reducer.sum(), geometry=roi,
            scale=375, maxPixels=1e9, bestEffort=True
        ).get("T21")
        return ee.Feature(None, {"date": date, "fire_pixels": count})
    
    records = collection.map(count_pixels).getInfo()["features"]
    df = pd.DataFrame([{"date": r["properties"]["date"],
                        "fire_pixels": r["properties"].get("fire_pixels", 0) or 0}
                       for r in records])
    df["date"] = pd.to_datetime(df["date"])
    return df.set_index("date").sort_index()

print("Extracting fire pixel counts by day (30-60s)...")
df_base = get_monthly_fire_counts(firms_baseline_col, ROI, FIRMS_BRIGHTNESS_K)
df_rec  = get_monthly_fire_counts(firms_recent_col,   ROI, FIRMS_BRIGHTNESS_K)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=False)
fig.suptitle("PipelineWatch-NG — VIIRS Fire Hotspot Time Series
Trans Niger Pipeline corridor",
             fontsize=13)

for ax, df, color, label in [
    (axes[0], df_base, "#378ADD", f"Baseline ({BASELINE_START} to {BASELINE_END})"),
    (axes[1], df_rec,  "#E24B4A", f"Recent   ({RECENT_START} to {RECENT_END})"),
]:
    rolled = df["fire_pixels"].rolling(7, min_periods=1).mean()
    ax.bar(df.index, df["fire_pixels"], color=color, alpha=0.3, width=1)
    ax.plot(df.index, rolled, color=color, linewidth=2, label="7-day mean")
    ax.axhline(df["fire_pixels"].mean(), color=color, linewidth=1,
               linestyle="--", alpha=0.7)
    ax.set_ylabel("Hot pixels")
    ax.set_title(label, fontsize=11)
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
os.makedirs("../outputs", exist_ok=True)
plt.savefig("../outputs/m1_fire_timeseries.png", dpi=150, bbox_inches="tight")
print("Saved: outputs/m1_fire_timeseries.png")
plt.show()

## Cell 10 — SO₂ baseline vs recent comparison

In [ ]:
import numpy as np

def get_so2_stats(composite, roi):
    return composite.select("SO2_mean_DU").reduceRegion(
        reducer=ee.Reducer.mean()
                          .combine(ee.Reducer.max(), sharedInputs=True)
                          .combine(ee.Reducer.percentile([50, 75]), sharedInputs=True),
        geometry=roi, scale=5500, maxPixels=1e9, bestEffort=True
    ).getInfo()

print("Computing SO2 statistics...")
base_stats = get_so2_stats(so2_baseline, ROI)
rec_stats  = get_so2_stats(so2_recent, ROI)

metrics   = ["Mean", "Median (p50)", "p75", "Max"]
keys      = ["SO2_mean_DU_mean", "SO2_mean_DU_p50",
             "SO2_mean_DU_p75", "SO2_mean_DU_max"]
base_vals = [base_stats.get(k, 0) or 0 for k in keys]
rec_vals  = [rec_stats.get(k, 0)  or 0 for k in keys]

x, w = np.arange(len(metrics)), 0.35
fig, ax = plt.subplots(figsize=(9, 5))
fig.suptitle("TROPOMI SO2 — Baseline vs Recent
Trans Niger Pipeline corridor", fontsize=12)

ax.bar(x - w/2, base_vals, w, label=f"Baseline ({BASELINE_START[:7]} to {BASELINE_END[:7]})",
       color="#378ADD", alpha=0.85)
ax.bar(x + w/2, rec_vals,  w, label=f"Recent   ({RECENT_START[:7]} to {RECENT_END[:7]})",
       color="#E24B4A", alpha=0.85)
ax.axhline(SO2_THRESHOLD_DU, color="#854F0B", linewidth=1.5, linestyle="--",
           label=f"Anomaly threshold ({SO2_THRESHOLD_DU} DU)")
ax.set_ylabel("SO2 column density (Dobson Units)")
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)

for bars in [ax.patches[:4], ax.patches[4:8]]:
    for bar in bars:
        h = bar.get_height()
        if h > 0.01:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.03, f"{h:.2f}",
                    ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("../outputs/m1_so2_comparison.png", dpi=150, bbox_inches="tight")
print("Saved: outputs/m1_so2_comparison.png")
plt.show()

## Cell 11 — Export GeoJSON for dashboard

In [ ]:
import json
from datetime import datetime

CACHE_DIR = "../data/cached"
os.makedirs(CACHE_DIR, exist_ok=True)

def export_fc(fc, filename, label, max_features=500):
    path = os.path.join(CACHE_DIR, filename)
    n = fc.size().getInfo()
    if n == 0:
        print(f"  {label}: 0 features")
        with open(path, "w") as f:
            json.dump({"type": "FeatureCollection", "features": []}, f)
        return
    print(f"  Exporting {label}: {n} features...")
    gj = fc.limit(max_features).getInfo()
    gj["metadata"] = {"exported": datetime.now().isoformat(),
                      "count": n, "signal": label,
                      "period": f"{RECENT_START} to {RECENT_END}"}
    with open(path, "w") as f:
        json.dump(gj, f, indent=2)
    print(f"    -> {path}  ({os.path.getsize(path)/1024:.1f} KB)")

print("=== Exporting GeoJSON outputs ===")
export_fc(dark_spots,    "sar_dark_spots.geojson",  "SAR dark spots")
export_fc(fire_hotspots, "fire_hotspots.geojson",   "VIIRS fire hotspots")
export_fc(so2_anomalies, "so2_anomalies.geojson",   "TROPOMI SO2 anomalies")

meta = {"module": "M1_ingestion",
        "exported": datetime.now().isoformat(),
        "study_area": "Trans Niger Pipeline corridor, Niger Delta",
        "bbox": ROI_BOUNDS,
        "detections": {"sar_dark_spots": n_spots,
                       "fire_hotspots":  n_hotspots,
                       "so2_anomalies":  n_so2}}
with open(os.path.join(CACHE_DIR, "m1_metadata.json"), "w") as f:
    json.dump(meta, f, indent=2)

print("\nAll outputs written to data/cached/ — ready to commit to GitHub.")

## Cell 12 — Module 1 complete

| Output | File | Description |
|--------|------|-------------|
| SAR dark spots |  | Oil spill candidate polygons |
| Fire hotspots |  | Persistent VIIRS fire locations |
| SO₂ anomalies |  | TROPOMI chemical plume zones |
| Fire time series |  | Baseline vs recent |
| SO₂ comparison |  | Baseline vs recent SO₂ |

**Next: Module 2** — Sentinel-2 NDVI/NDWI vegetation dieback · DBSCAN refinery clustering · SAR change detection